# Notebook 3: Testing Modified SGP4 on LAGEOS-1


Note that the positional and velocity outputs below are in the True Equator Mean Equinox (TEME) reference frame according to [SGP4 PyPI documentation](https://pypi.org/project/sgp4/), so this will need to be converted to GCRF to be compatible with SP3 data for residual calculation.

In [ ]:
# Imports and paths
import os
import sys
import subprocess
import numpy as np
import pandas as pd
from pathlib import Path

os.environ["JAVA_HOME"] = str(Path(sys.prefix) / "Library" / "jre")

from brent import Constants
from brent.propagators.analytical.tle import (
    TLEPropagator, TLEPropagatorMethod
)

repo_path = Path.cwd().resolve()
if repo_path.name == "experiments":
    repo_path = repo_path.parent

data_path = repo_path / "data"
download_path = repo_path / "download"

## Comparing Original \& Modified SGP4 Implementation:

I want to compare the unmodified SGP4 implementation in the branch `sgp4-baseline` with my current branch `prelim-experiments` with the modified implementation. This will be the final test before doing the changes to $C_X$.

In [ ]:
# CHANGE THIS WHEN SWITCHING BRANCHES
RUN_LABEL = "main"  # "main" or "hujsak"

branch = subprocess.check_output(
    ["git", "branch", "--show-current"],
    cwd=repo_path,
    text=True
).strip()

print("Current branch:", branch)
print("Output label:", RUN_LABEL)

# Load the existing SP3 timestamps
sp3_df = pd.read_csv(
    data_path / "states" / "lageos1_sp3_2025.csv",
    parse_dates=["epoch_utc"]
)

dates = sp3_df["epoch_utc"]

# Load LAGEOS-1 TLEs using BRENT
tle = TLEPropagator.load(
    path=str(download_path / "tle" / "8820.json"),
    method=TLEPropagatorMethod.FORWARD
)

# Propagate using the active branch's SGP4 implementation
states = np.array([
    np.asarray(
        tle._propagate(date.to_pydatetime(), Constants.DEFAULT_ECI),
        dtype=float
    )
    for date in dates
])

assert states.shape == (len(dates), 6)
assert np.isfinite(states).all()

columns = [
    "x_m", "y_m", "z_m",
    "vx_m_s", "vy_m_s", "vz_m_s"
]

result = pd.DataFrame(states, columns=columns)
result.insert(0, "epoch_utc", dates)

output = data_path / "states" / f"lageos1_tle_2025_{RUN_LABEL}.csv"
result.to_csv(output, index=False)

print(f"Saved {len(result)} states to {output.name}")